## Analysis of Russia-China border infrastructure and human activity over time

# Install and import modules

In [7]:
%pip install earthengine-api geemap cartopy seaborn pandas matplotlib

import ee
import geemap
import pandas as pd
import matplotlib as plt


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Initialise

In [8]:
ee.Authenticate()
ee.Initialize(project='alphaearth-468711')

# Russia-China border geometry
russiaChinaBorderObject  = geemap.geojson_to_ee('../source-files/china-russia-border.geojson')
russiaChinaBorderGeom = russiaChinaBorderObject.geometry()
russiaChinaBorderGeomBuffered = russiaChinaBorderGeom.buffer(100000) # buffer by 50km

# Night lights

In [ ]:
#focus cities along border
cities = ['Heihe', 'Khabarovsk', 'Manzhouli', 'Vladivostok']

# initialise nightlight dataset
nightlights = ee.ImageCollection('NASA/VIIRS/002/VNP46A2')
latest = nightlights.sort('system:time_start', False).first()
print('Latest image date: ' + latest.date().format().getInfo())

nightlights_filtered = nightlights.filter(ee.Filter.date('2017-01-01', latest.date())).select('Gap_Filled_DNB_BRDF_Corrected_NTL')
nighttimeVis = {min: 0.0, max: 1.0};


Latest image date: 2026-11-02T00:00:00


In [11]:
# --- Scaled nightlights (no masking) ------------------------------------
# VNP46A2 NTL bands are stored with a 0.1 scale factor -> nW/cm2/sr
SCALE = 0.1

# Season filter (not a mask - it just selects which nights go into the median).
# Set SEASON = None to use the whole year; winter snow will then inflate
# radiance at this latitude.
SEASON = ee.Filter.calendarRange(5, 9, 'month')   # May-September

def to_ntl(img):
    return (img.select('Gap_Filled_DNB_BRDF_Corrected_NTL')
               .multiply(SCALE)
               .rename('NTL')
               .copyProperties(img, ['system:time_start']))

ntl = nightlights.filter(SEASON) if SEASON else nightlights
ntl = ntl.map(to_ntl)

def composite(start, end):
    return ntl.filter(ee.Filter.date(start, end)).median().rename('NTL')

In [12]:
# --- Baseline vs recent -------------------------------------------------
baseNTL   = composite('2017-01-01', '2018-01-01')
recentNTL = composite('2025-01-01', '2026-01-01')

abs_change = recentNTL.subtract(baseNTL).rename('abs_change')

# Denominator floor (nW/cm2/sr). Not a mask - nothing is hidden - it just stops
# near-zero background pixels from turning sensor noise into +/-1000%.
# Set DENOM_FLOOR = 0 for the raw ratio.
DENOM_FLOOR = 0.5

pct_change = (recentNTL.subtract(baseNTL)
                .divide(baseNTL.max(DENOM_FLOOR))
                .multiply(100)
                .rename('pct_change'))

buf = russiaChinaBorderGeomBuffered

In [ ]:
# --- Map ----------------------------------------------------------------
# Log stretch on the reference layer: settlement radiance spans 3+ orders of
# magnitude, so a linear stretch shows either cities or villages, never both.
log_recent = recentNTL.max(0.05).log10()
log_base   = baseNTL.max(0.05).log10()

ntlVis    = {'min': -1.3, 'max': 1.7, 'palette': ["#2c0b54",'#3b0f70','#8c2981','#de4968','#fe9f6d','#fcfdbf']}
changeVis = {'min': -100, 'max': 100, 'palette': ['#2166ac', '#f7f7f7', '#b2182b']}
absVis    = {'min': -5,   'max': 5,   'palette': ['#2166ac', '#f7f7f7', '#b2182b']}

NightlightsMap = geemap.Map(center=[48, 128], zoom=5)

# English-language basemap. The default OSM basemap labels places in the local
# script (Cyrillic / Chinese); Google's tiles honour an hl= language parameter.
NightlightsMap.clear_layers()
NightlightsMap.add_tile_layer(
    url='https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}&hl=en',
    name='Google Roads (EN)',
    attribution='Google',
)

NightlightsMap.addLayer(log_recent.clip(buf),  ntlVis,    'NTL 2025 (log10)')
NightlightsMap.addLayer(log_base.clip(buf),  ntlVis,    'NTL 2017 (log10)', False)
NightlightsMap.addLayer(abs_change.clip(buf),  absVis,    'Absolute NTL change', False)
NightlightsMap.addLayer(pct_change.clip(buf),  changeVis, '% NTL change 2017 -> 2025', True)
NightlightsMap.addLayer(buf, {}, 'Border buffer (50km)', False)
NightlightsMap.addLayer(russiaChinaBorderGeom, {}, 'Border', True)
NightlightsMap

Map(center=[48, 128], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tra…

### Clip nightlights around selected cities

In [15]:
import os, json, requests

EXPORT_DIR = '../exports'
os.makedirs(EXPORT_DIR, exist_ok=True)

RADIUS_M = 50000     # 50 km
EXPORT_SCALE = 463.83  # native VNP46A2 pixel size (~15 arc-sec), metres

# City coordinates from the populated-places source file
with open('../source-files/populated-places-clipped.geojson', encoding='utf-8') as f:
    places = json.load(f)

city_pts = {}
for feat in places['features']:
    name = feat['properties']['NAME']
    if name in cities:
        city_pts[name] = feat['geometry']['coordinates']  # [lon, lat]
missing = set(cities) - set(city_pts)
assert not missing, f'Not found in geojson: {missing}'

layers = {'NTL_2025': recentNTL, 'NTL_2017': baseNTL}
png_vis = {'min': ntlVis['min'], 'max': ntlVis['max'], 'palette': ntlVis['palette']}

for city in cities:
    lon, lat = city_pts[city]
    region = ee.Geometry.Point([lon, lat]).buffer(RADIUS_M)
    box = region.bounds()
    for lname, img in layers.items():
        clipped = img.clip(region)

        # GeoTIFF: raw radiance (nW/cm2/sr)
        tif = f'{EXPORT_DIR}/{city}_{lname}.tif'
        geemap.ee_export_image(
            clipped, filename=tif, scale=EXPORT_SCALE, region=box,
            crs='EPSG:4326', file_per_band=False)

        # PNG: log10 stretch, same as map; pixels outside the 50 km circle are transparent
        png = f'{EXPORT_DIR}/{city}_{lname}.png'
        url = (clipped.max(0.05).log10()
               .visualize(**png_vis)
               .getThumbURL({'region': box, 'dimensions': 1024, 'format': 'png'}))
        r = requests.get(url, timeout=300)
        r.raise_for_status()
        with open(png, 'wb') as f:
            f.write(r.content)

        print('saved', tif, '|', png)

print(sorted(os.listdir(EXPORT_DIR)))


Generating URL ...
Please wait ...
Data downloaded to /Users/josephclark/Documents/Birkbeck/Dissertation/python-files/exports/Heihe_NTL_2025.tif
saved ../exports/Heihe_NTL_2025.tif | ../exports/Heihe_NTL_2025.png
Generating URL ...
Please wait ...
Data downloaded to /Users/josephclark/Documents/Birkbeck/Dissertation/python-files/exports/Heihe_NTL_2017.tif
saved ../exports/Heihe_NTL_2017.tif | ../exports/Heihe_NTL_2017.png
Generating URL ...
Please wait ...
Data downloaded to /Users/josephclark/Documents/Birkbeck/Dissertation/python-files/exports/Khabarovsk_NTL_2025.tif
saved ../exports/Khabarovsk_NTL_2025.tif | ../exports/Khabarovsk_NTL_2025.png
Generating URL ...
Please wait ...
Data downloaded to /Users/josephclark/Documents/Birkbeck/Dissertation/python-files/exports/Khabarovsk_NTL_2017.tif
saved ../exports/Khabarovsk_NTL_2017.tif | ../exports/Khabarovsk_NTL_2017.png
Generating URL ...
Please wait ...
Data downloaded to /Users/josephclark/Documents/Birkbeck/Dissertation/python-files/